# 04 · Scale labels, tune fastText, review coverage (Lecture 5)

**Question:** with about 1,000 reviewed labels, how accurate is a tuned fastText model, where does it fail, and
how much can it label without human review?

Lecture 5 asked for 500 to 1,000 reviewed labels per group with AI suggestions plus human review, fastText
retrained and tuned on validation (features, then epochs and learning rate, then vector size), per-class errors,
a review cutoff chosen on validation, and the effort to scale.

Our PA5 work: Kerim's notebook built the labeling loop (900 new memos, rule suggestions with quoted cues, timed
review, 25% audit; 14% of the audit needed correcting, so every row was reviewed by hand) and the evaluation;
Aiden's notebook added the gambling rules, the character n-gram recheck and the seen/unseen pattern analysis.

### Growing the label set (run on DataHub, in a terminal)
```bash
python -m src.main --config config.yaml --steps sample        # freeze split, choose batch, AI suggestions
python -m src.review --config config.yaml                     # timed human review
python -m src.main --config config.yaml --steps audit,build-labels
```
Each batch over-samples Subscriptions & Entertainment (`sampling.focus_category`) so the Project A category has
enough training rows in every group, not mostly gambling.

> Runs on DataHub. Cells show aggregate tables and charts only; files with memo text (errors, examples, predictions) stay in `outputs/` and are never printed here or committed.

In [ ]:
import os, sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
CONFIG = os.environ.get("CAPSTONE_CONFIG", "config.yaml")

import pandas as pd
from IPython.display import Image, display
from src.config import load_config
from src.main import run
OUT = load_config(CONFIG)["paths"]["out_dir"]
pd.set_option("display.max_colwidth", 80)

## Tuning (validation only) and the final model

In [ ]:
m = run(CONFIG, "prepare,train")
m["model"]["params"], m["model"]["text_col"], m["model"]["fit_note"]

In [ ]:
exp = pd.read_csv(OUT / "experiments.csv")
exp.groupby("stage")[["macro_f1", "accuracy"]].max()

In [ ]:
pd.read_csv(OUT / "stability.csv")

## Class quality, review coverage and Test A / Test B

In [ ]:
m = run(CONFIG, "evaluate")
pd.read_csv(OUT / "scores.csv")

In [ ]:
pd.read_csv(OUT / "per_class_val.csv", index_col=0)

In [ ]:
display(Image(OUT / "confusion.png"))

In [ ]:
print(m["evaluate"]["cutoff"]["why"])
pd.read_csv(OUT / "coverage_val.csv")

In [ ]:
pd.read_csv(OUT / "class_mix.csv", index_col=0)

In [ ]:
pd.DataFrame(m["evaluate"]["gambling_slice"])

Read the Test A vs Test B gap next to the class mix and the gambling slice: in PA5, Test B's gambling
rows (mostly two accounts, brands unseen in training) scored far below its other rows.

## Effort to scale (Participation question 1)

In [ ]:
m = run(CONFIG, "runtime")
{k: v for k, v in m["runtime"].items() if k != "hardware"}